In [1]:
!uv pip install mantis-tsfm catboost lightgbm

Using Python 3.13.2 environment at: /home/petelin/TSCGlue/.venv
Audited 3 packages in 14ms


In [2]:
import torch
import torch.nn.functional as F
import numpy as np
from sklearn.metrics import accuracy_score
from mantis.architecture import MantisV2
from mantis.trainer import MantisTrainer
from tscglue import data_loader

In [3]:
device = 'cpu'
network = MantisV2(device=device)
network = network.from_pretrained("paris-noah/MantisV2")
model = MantisTrainer(device=device, network=network)


In [4]:
dataset = "Worms"
dataset = 'Car'
# dataset = 'HandOutlines'
# dataset = 'Trace'
dataset = 'SwedishLeaf'
dataset = 'FordA'
X_train, y_train, X_test, y_test = data_loader.load_fold(dataset, 0)
print(X_train.shape, y_train.shape, X_test.shape, y_test.shape)

(3601, 1, 500) (3601,) (1320, 1, 500) (1320,)


In [5]:
def resize(X, size=512):
    return F.interpolate(
        torch.tensor(X, dtype=torch.float), size=size, mode='linear', align_corners=False
    ).numpy()

X_train_r = resize(X_train)
X_test_r = resize(X_test)
print(X_train_r.shape, X_test_r.shape)

(3601, 1, 512) (1320, 1, 512)


In [6]:
# Extract frozen embeddings (no fine-tuning)
Z_train = model.transform(X_train_r)
Z_test = model.transform(X_test_r)
print(Z_train.shape, Z_test.shape)

(3601, 256) (1320, 256)


In [7]:
# Classify on frozen embeddings
from sklearn.linear_model import RidgeClassifierCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

pipe = make_pipeline(StandardScaler(), RidgeClassifierCV(alphas=np.logspace(-3, 3, 10)))
pipe.fit(Z_train, y_train)
y_pred = pipe.predict(Z_test)
print(f"Mantis embeddings + RidgeCV accuracy: {accuracy_score(y_test, y_pred):.4f}")

Mantis embeddings + RidgeCV accuracy: 0.9318


In [8]:
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier

for name, clf in [
    ("RidgeCV", RidgeClassifierCV(alphas=np.logspace(-3, 3, 10))),
    ("RandomForest", RandomForestClassifier(n_estimators=500, random_state=42, n_jobs=-1)),
    ("ExtraTrees", ExtraTreesClassifier(n_estimators=500, random_state=42, n_jobs=-1)),
    ("HistGradientBoosting", HistGradientBoostingClassifier(max_iter=500, random_state=42)),
    ("LightGBM", LGBMClassifier(n_estimators=500, random_state=42, verbose=-1)),
    ("CatBoost", CatBoostClassifier(iterations=500, random_seed=42, verbose=0)),
]:
    pipe = make_pipeline(StandardScaler(), clf)
    pipe.fit(Z_train, y_train)
    y_pred = pipe.predict(Z_test)
    print(f"Mantis embeddings + {name}: {accuracy_score(y_test, y_pred):.4f}")

Mantis embeddings + RidgeCV: 0.9318
Mantis embeddings + RandomForest: 0.9076
Mantis embeddings + ExtraTrees: 0.9015
Mantis embeddings + HistGradientBoosting: 0.9189


/home/petelin/TSCGlue/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Mantis embeddings + LightGBM: 0.9129
Mantis embeddings + CatBoost: 0.9114


In [9]:
Z_train.shape

(3601, 256)

In [ ]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

model.fit(X_train_r, y_train_enc, num_epochs=8)
y_pred = le.inverse_transform(model.predict(X_test_r))
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")

Epoch 7: Train Loss 0.1451: 100%|██████████| 8/8 [13:12<00:00, 99.08s/it] 


In [ ]:
from chronos import BaseChronosPipeline

chronos_pipeline = BaseChronosPipeline.from_pretrained("amazon/chronos-2", device_map="cpu")

In [ ]:
def chronos_embed(pipeline, X, batch_size=64):
    all_embs = []
    for i in range(0, len(X), batch_size):
        batch = [torch.from_numpy(x.squeeze(0)).float() for x in X[i:i+batch_size]]
        embeddings, _ = pipeline.embed(batch)
        # [REG] token at position [-2] per series
        vecs = [e[0, -2, :].detach().cpu().numpy() for e in embeddings]
        all_embs.append(np.stack(vecs))
    return np.vstack(all_embs)

Z_train_c = chronos_embed(chronos_pipeline, X_train)
Z_test_c = chronos_embed(chronos_pipeline, X_test)
print(Z_train_c.shape, Z_test_c.shape)

In [ ]:
for name, clf in [
    ("RidgeCV", RidgeClassifierCV(alphas=np.logspace(-3, 3, 10))),
    ("RandomForest", RandomForestClassifier(n_estimators=500, random_state=42, n_jobs=-1)),
    ("ExtraTrees", ExtraTreesClassifier(n_estimators=500, random_state=42, n_jobs=-1)),
    ("HistGradientBoosting", HistGradientBoostingClassifier(max_iter=500, random_state=42)),
    ("LightGBM", LGBMClassifier(n_estimators=500, random_state=42, verbose=-1)),
    ("CatBoost", CatBoostClassifier(iterations=500, random_seed=42, verbose=0)),
]:
    pipe = make_pipeline(StandardScaler(), clf)
    pipe.fit(Z_train_c, y_train)
    y_pred = pipe.predict(Z_test_c)
    print(f"Chronos2 embeddings + {name}: {accuracy_score(y_test, y_pred):.4f}")

In [ ]:
# Combine Mantis (256d) + Chronos2 (768d) embeddings
Z_train_combined = np.hstack([Z_train, Z_train_c])
Z_test_combined = np.hstack([Z_test, Z_test_c])
print(Z_train_combined.shape, Z_test_combined.shape)

for name, clf in [
    ("RidgeCV", RidgeClassifierCV(alphas=np.logspace(-3, 3, 10))),
    ("RandomForest", RandomForestClassifier(n_estimators=500, random_state=42, n_jobs=-1)),
    ("ExtraTrees", ExtraTreesClassifier(n_estimators=500, random_state=42, n_jobs=-1)),
    ("HistGradientBoosting", HistGradientBoostingClassifier(max_iter=500, random_state=42)),
    ("LightGBM", LGBMClassifier(n_estimators=500, random_state=42, verbose=-1)),
    ("CatBoost", CatBoostClassifier(iterations=500, random_seed=42, verbose=0)),
]:
    pipe = make_pipeline(StandardScaler(), clf)
    pipe.fit(Z_train_combined, y_train)
    y_pred = pipe.predict(Z_test_combined)
    print(f"Mantis+Chronos2 embeddings + {name}: {accuracy_score(y_test, y_pred):.4f}")

In [ ]:
from tscglue.models_tsfm import ALL_TSFM_MODELS, make_tsfm_model

for model_name in ALL_TSFM_MODELS:
    m = make_tsfm_model(model_name)
    m.fit(X_train, y_train)
    y_pred = m.predict(X_test)
    print(f"{model_name}: {accuracy_score(y_test, y_pred):.4f}")

2026-03-10 08:03:23.767011: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


mantis-ridgecv: 0.9318
mantis-rf: 0.9076
mantis-et: 0.9015
mantis-hgb: 0.9189


/home/petelin/TSCGlue/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


mantis-lgbm: 0.9129
mantis-catboost: 0.9114
chronos2-ridgecv: 0.9326
chronos2-rf: 0.8492
chronos2-et: 0.8553
chronos2-hgb: 0.8992


/home/petelin/TSCGlue/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


chronos2-lgbm: 0.8939
chronos2-catboost: 0.8894
